In [ ]:
from openai import OpenAI
from tqdm import tqdm
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def process_data(input_file, output_file, prompt='Complete the sentence. Answer with one word.'):
  df = pd.read_csv(input_file)
  if "prompt" not in df.columns:
    df["prompt"] = None
  if "model_output" not in df.columns:
    df["model_output"] = None

  client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="<OPENROUTER_API_KEY>",
  )

  for i in tqdm(df.index):
    if pd.notna(df.at[i, "model_output"]):  # пропуск готовых
      continue

    response = client.chat.completions.create(
    model="deepseek/deepseek-v4-flash",
    messages=[
          {
            "role": "user",
            "content": f'{prompt} "{df.at[i, "query"]}"'
          }
        ],
      extra_body={"reasoning": {"enabled": False}}
    )

    response = response.choices[0].message
    df.at[i, "model_output"]

    df.at[i, "prompt"] = messages[0]["content"]

    if i % 100 == 0:  # периодическое сохранение
            df.to_csv(output_file, index=False)

  df.to_csv(output_file, index=False)
  return df

In [ ]:
import asyncio
from openai import AsyncOpenAI

In [ ]:
OPENROUTER_API_KEY = ""


async def process_df_async_detail(
    input_file,
    concurrency=8,
    save_every=5,
    save_path=None,
):
    prompt='Complete the sentence. Answer with one word.'
    df = pd.read_csv(input_file)
    if "prompt" not in df.columns:
      df["prompt"] = None
    if "model_output" not in df.columns:
      df["model_output"] = None

    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    sem = asyncio.Semaphore(concurrency)

    async def process_one(i):
        async with sem:
            row = df.loc[i]

            try:
                content = f'{prompt} "{df.at[i, "query"]}"'
                response = await client.chat.completions.create(
                    model="deepseek/deepseek-v4-flash",
                    messages=[
                        {
                            "role": "user",
                            "content": content,
                        }
                    ],
                    extra_body={"reasoning": {"enabled": True}}
                )

                response = response.choices[0].message
                model_output = response.content
                return i, model_output, content

            except Exception as e:
                print(f"Ошибка в строке {i}: {e}")
                used_prompt = f'{prompt} "{df.at[i, "query"]}"'
                return i, None, content

    tasks = [process_one(i) for i in df.index]
    done_count = 0

    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        index, output, used_prompt = await fut

        df.at[index, "model_output"] = output
        df.at[index, "prompt"] = used_prompt
        done_count += 1

        if save_path and done_count % save_every == 0:
            df.to_csv(save_path, index=False)

    if save_path:
        df_to_save = df.drop(columns=["image_data_url"], errors="ignore")
        df_to_save.to_csv(save_path, index=False)

    return df

In [ ]:

result = await process_df_async_detail(
    "/content/drive/MyDrive/диплом/select_pararel.csv",
    concurrency=8,
    save_path="/content/drive/MyDrive/диплом/neutral_res_deepseek.csv"
)

100%|██████████| 1250/1250 [13:39<00:00,  1.52it/s]


In [ ]:
result.head()

,subject,rel_lemma,object,rel_p_id,query,prompt,model_output
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,"Complete the sentence. Answer with one word. ""...",Hawaiian
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,"Complete the sentence. Answer with one word. ""...",Russian
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,"Complete the sentence. Answer with one word. ""...",Dutch
3,Symeon of Polotsk,mother-tongue,Russian,P37,The mother tongue of Symeon of Polotsk is,"Complete the sentence. Answer with one word. ""...",Belarusian
4,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,"Complete the sentence. Answer with one word. ""...",Finnish


In [ ]:
neutral_res = pd.read_csv("/content/drive/MyDrive/диплом/neutral_res_deepseek.csv")

In [ ]:
neutral_res.head()

,subject,rel_lemma,object,rel_p_id,query,prompt,model_output
0,Hawaii,the-official-language,English,P37,The official language of Hawaii is,"Complete the sentence. Answer with one word. ""...",Hawaiian
1,Oleg Skripochka,mother-tongue,Russian,P37,The mother tongue of Oleg Skripochka is,"Complete the sentence. Answer with one word. ""...",Russian
2,Flemish Community,the-official-language,Dutch,P37,The official language of Flemish Community is,"Complete the sentence. Answer with one word. ""...",Dutch
3,Symeon of Polotsk,mother-tongue,Russian,P37,The mother tongue of Symeon of Polotsk is,"Complete the sentence. Answer with one word. ""...",Belarusian
4,Nousiainen,the-official-language,Finnish,P37,The official language of Nousiainen is,"Complete the sentence. Answer with one word. ""...",Finnish


In [ ]:
print(f"P37: {(neutral_res[neutral_res["rel_p_id"]=="P37"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P30: {(neutral_res[neutral_res["rel_p_id"]=="P30"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P1376: {(neutral_res[neutral_res["rel_p_id"]=="P1376"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P27: {(neutral_res[neutral_res["rel_p_id"]=="P27"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")
print(f"P279: {(neutral_res[neutral_res["rel_p_id"]=="P279"].apply(lambda x: x["object"].lower() in x["model_output"].lower(), axis=1)).sum()}")

P37: 233
P30: 244
P1376: 237
P27: 235
P279: 204


In [ ]:
filtered_deepseek = neutral_res[
    (neutral_res.apply(lambda x: str(x["object"]).lower() in str(x["model_output"]).lower(), axis=1)) &
    (neutral_res["rel_p_id"] != "P279")
].copy()

In [ ]:
filtered_deepseek.shape

(949, 7)

In [ ]:
filtered_deepseek["consistent_sent"] = filtered_deepseek["query"] + " " + filtered_deepseek["object"]
filtered_deepseek["conflict_sent"] = None

In [ ]:
import random

In [ ]:
filtered_deepseek["sub_object"] = None
for i in filtered_deepseek.index:
  id = filtered_deepseek.at[i, "rel_p_id"]
  sub_options = list(set(filtered_deepseek[filtered_deepseek["rel_p_id"] == id]["object"].unique().tolist()) - set([filtered_deepseek.at[i, "object"]]))
  sub_object = random.choice(sub_options)
  filtered_deepseek.at[i, "sub_object"] = sub_object
  filtered_deepseek.at[i, "conflict_sent"] = filtered_deepseek.at[i, "query"] + " " + sub_object


In [ ]:
filtered_deepseek = filtered_deepseek.drop(["prompt", "model_output"], axis=1)

In [ ]:
filtered_deepseek.to_csv("/content/drive/MyDrive/диплом/deepseek_context_sents.csv", index=False)

In [ ]:
async def process_df_async(
    input_file,
    concurrency=8,
    save_every=5,
    exp_n=1,
    prompt="neutral",
    confidence="percents",
    save_path=None,
):
    df = pd.read_csv(input_file)
    if "prompt" not in df.columns:
      df["prompt"] = None
    if f"model_output_{exp_n}" not in df.columns:
      df[f"model_output_{exp_n}"] = None

    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    sem = asyncio.Semaphore(concurrency)

    async def process_one(i):
        async with sem:
            if prompt == "neutral":
              prompt_text = "Complete the sentence. Answer with one word."
            elif prompt == "consistent":
              prompt_text = df.at[i, "consistent_sent"] + ". Complete the sentence. "
            elif prompt == "conflict":
              prompt_text = df.at[i, "conflict_sent"] + ". Complete the sentence. "
            elif prompt == "conflict_confident_user":
              prompt_text = "I believe, that " + df.at[i, "conflict_sent"] + ". Complete the sentence. "

            if confidence == "percents":
              prompt_text += """In addition, estimate, how confident are you in given answer from 0 to 100, where 0 is absolutely not confident and 100 is absolutely confident.
              Return only two items: the answer (sentence completion) and number (confidence estimation) separated by space without any other formatting. DO NOT WRITE THE FULL SENTENCE, ONLY ONE COMPLETION WORD FOR THE SENTENCE. Example - Sentence: The dog belongs to the family of Answer: mammals 100. """
            elif confidence == "verbalized":
              prompt_text += """In addition, estimate, how confident are you in given answer.
              Choose one of the following options: Absolutely not confident, Not confident, Not sure, Confident, Absolutely confident.
              Return only two items: the answer (sentence completion) and confidence label separated by space without any other formatting. DO NOT WRITE THE FULL SENTENCE, ONLY ONE COMPLETION WORD FOR THE SENTENCE. Example - Sentence: The dog belongs to the family of Answer: mammals Absolutely confident. """
            elif confidence == "none":
              prompt_text += """Return only one item: the answer (sentence completion) without any other formatting. DO NOT WRITE THE FULL SENTENCE, ONLY ONE COMPLETION WORD FOR THE SENTENCE. Example - Sentence: The dog belongs to the family of Answer: mammals"""


            prompt_text += f"Your sentence: {df.at[i, 'query']}"

            try:
                content = prompt_text
                response = await client.chat.completions.create(
                    model="deepseek/deepseek-v4-flash",
                    messages=[
                        {
                            "role": "user",
                            "content": prompt_text,
                        }
                    ],
                    extra_body={"reasoning": {"enabled": True}}
                )

                response = response.choices[0].message
                model_output = response.content
                return i, model_output, content

            except Exception as e:
                print(f"Ошибка в строке {i}: {e}")
                used_prompt = f'{prompt} "{df.at[i, "query"]}"'
                return i, None, content

    tasks = [process_one(i) for i in df.index]
    done_count = 0

    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        index, output, used_prompt = await fut

        df.at[index, f"model_output_{exp_n}"] = output
        df.at[index, "prompt"] = used_prompt
        done_count += 1

        if save_path and done_count % save_every == 0:
            df.to_csv(save_path, index=False)

    if save_path:
        df.to_csv(save_path, index=False)

    return df

In [ ]:
ds_n_p = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="neutral",
    confidence="percents",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_neutral_percents.csv"
)

100%|██████████| 949/949 [07:57<00:00,  1.99it/s]


In [ ]:
ds_n_p = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="neutral",
    confidence="verbalized",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_neutral_verbalized.csv"
)

100%|██████████| 949/949 [06:28<00:00,  2.44it/s]


In [ ]:
ds_cons_p = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="consistent",
    confidence="percents",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_consistent_percents.csv")

100%|██████████| 949/949 [14:36<00:00,  1.08it/s]


In [ ]:
ds_cons_v = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="consistent",
    confidence="verbalized",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_consistent_verbalized.csv")

100%|██████████| 949/949 [10:21<00:00,  1.53it/s]


In [ ]:
ds_cons_p = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="conflict",
    confidence="percents",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_conflict_percents.csv")

100%|██████████| 949/949 [35:02<00:00,  2.22s/it]


In [ ]:
ds_cons_p = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="conflict",
    confidence="verbalized",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_conflict_verbalized.csv")

100%|██████████| 949/949 [39:53<00:00,  2.52s/it] 


In [ ]:
ds_conf_user_p = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="conflict_confident_user",
    confidence="percents",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_conflict_confident_user_percents.csv")

100%|██████████| 949/949 [39:56<00:00,  2.53s/it]


In [ ]:
ds_conf_user_p = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="conflict_confident_user",
    confidence="verbalized",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_conflict_confident_user_verbalized.csv")

 27%|██▋       | 254/949 [07:36<29:02,  2.51s/it]

Ошибка в строке 518: Expecting value: line 217 column 1 (char 1188)


100%|██████████| 949/949 [30:35<00:00,  1.93s/it]


In [ ]:
ds_conf_none = await process_df_async(
    "/content/drive/MyDrive/диплом/deepseek_context_sents.csv",
    prompt="conflict",
    confidence="none",
    save_path="/content/drive/MyDrive/диплом/deepseek/deepseek_conflict_none.csv")

100%|██████████| 949/949 [32:54<00:00,  2.08s/it]
